# 22｜葡萄牙污水 Agilent DDA 批量转换 mzML 与质控

本笔记本按21号冻结的清单，将正、负离子模式共96个 Agilent `.d` 转为 mzML。每个输出文件都会解析至XML结尾，并检查MS1、MS2、极性、质心化和MS2前体元数据。

本步需要Windows 64位 ProteoWizard/MSConvert及其厂商数据读取器。转换逐文件执行，意外中断后可直接重跑，已通过质控的mzML会自动跳过。

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd
from IPython.display import display

ROOT = Path(r'E:\\谱图预测PBT_终版')
ENV_MS = Path(r'D:\\Appinstallation\\Anaconda\\envs\\env_ms\\python.exe')
SRC = ROOT / 'src'
CASE = ROOT / 'external_data/portugal_wastewater_5830725'
MANIFEST_DIR = CASE / 'manifests'

# 通常保持None，程序会自动查找msconvert.exe。
# 若自动查找失败，可填写完整路径，例如：
# MSCONVERT_EXE = r'C:\\Users\\Lenovo\\AppData\\Local\\Apps\\ProteoWizard 3.0.x 64-bit\\msconvert.exe'
MSCONVERT_EXE = None
RUN_CONVERSION = True
OVERWRITE_VALID_MZML = False

assert ENV_MS.is_file(), ENV_MS
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
sys.path.insert(0, str(SRC)) if str(SRC) not in sys.path else None
print('ROOT:', ROOT)
print('Notebook kernel:', sys.executable)
print(f'E drive free: {shutil.disk_usage(CASE).free / 1024**3:.1f} GiB')

## 1. 21号门控与MSConvert检测

21号必须显示 `READY_FOR_NOTEBOOK_22`。如未检测到 `msconvert.exe`，请从 [ProteoWizard官方下载页](https://proteowizard.sourceforge.io/download.html) 安装Windows 64位二进制版（含厂商读取器），重启Jupyter/PyCharm后再从头运行；也可在上一单元填写完整路径。

In [ ]:
from pbt_msms.portugal_msconvert import locate_msconvert, msconvert_version

nb21_path = MANIFEST_DIR / '21_portugal_dda_extraction_manifest.json'
nb21 = json.loads(nb21_path.read_text(encoding='utf-8'))
assert nb21.get('status') == 'READY_FOR_NOTEBOOK_22', nb21.get('status')
assert nb21.get('qc_all_required_pass') is True

resolved_msconvert = locate_msconvert(MSCONVERT_EXE)
if resolved_msconvert is None:
    raise FileNotFoundError(
        '未找到msconvert.exe。请安装Windows 64位ProteoWizard（包含厂商读取器），'
        '重启Jupyter/PyCharm后重跑22号，或在MSCONVERT_EXE中填入完整路径。'
    )
print('✓21号状态:', nb21['status'])
print('✓MSConvert:', resolved_msconvert)
print('✓Version:', msconvert_version(resolved_msconvert))

## 2. 批量转换

固定参数：`mzML + zlib + 64-bit`；过滤器顺序为 `peakPicking vendor msLevel=1-`、`zeroSamples removeExtra`、`titleMaker <RunId>.<ScanNumber>`。这是21号计划中 `peakPicking true 1-` 的显式现代写法；厂商峰提取必须是第一个过滤器。

为了保证可恢复性，每个数据集单独转换、单独写日志，通过XML和谱图级QC后才换成正式文件名。如首个文件失败，程序会立即停止，避免重复产96个同类错误。

In [ ]:
if RUN_CONVERSION:
    cmd = [str(ENV_MS), '-m', 'pbt_msms.portugal_msconvert',
           '--root', str(ROOT), '--msconvert', str(resolved_msconvert)]
    if OVERWRITE_VALID_MZML:
        cmd.append('--overwrite')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)
    print()
    print('✓ Notebook 22 批量转换执行完成')
else:
    print('RUN_CONVERSION=False：仅完成前置检查，没有转换。')

## 3. 最终QC

96个文件全部通过后，最终状态必须是 `READY_FOR_NOTEBOOK_23`。如转换中断，重新运行本笔记本即可，不要开启 `OVERWRITE_VALID_MZML`。

In [ ]:
manifest_path = MANIFEST_DIR / '22_msconvert_run_manifest.json'
status_path = MANIFEST_DIR / '22_msconvert_file_status.csv'
qc_path = MANIFEST_DIR / '22_msconvert_qc_checks.csv'

assert RUN_CONVERSION, '当前只做了前置检查，请将RUN_CONVERSION设为True后重跑。'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
status = pd.read_csv(status_path)
qc = pd.read_csv(qc_path)
display(pd.Series({
    'status': manifest['status'],
    'planned': manifest['counts']['planned'],
    'passed': manifest['counts']['passed'],
    'passed_by_mode': manifest['counts']['passed_by_mode'],
    'total spectra': int(status['spectra_n'].sum()),
    'total MS1': int(status['ms1_n'].sum()),
    'total MS2': int(status['ms2_n'].sum()),
    'mzML size (GiB)': round(status['mzml_size_bytes'].sum() / 1024**3, 3),
}, name='value').to_frame())
display(status.groupby('mode').agg(
    files=('dataset_id', 'size'), spectra=('spectra_n', 'sum'),
    ms1=('ms1_n', 'sum'), ms2=('ms2_n', 'sum'),
    min_ms2_precursor_fraction=('ms2_precursor_fraction', 'min'),
))
display(qc)
assert manifest['status'] == 'READY_FOR_NOTEBOOK_23', manifest['status']
assert manifest['qc_all_pass'] is True
assert len(status) == 96 and status['qc_pass'].astype(bool).all()
print()
print('✓最终状态:', manifest['status'])
print('下一步：23号进行mzML特征提取、DDA MS/MS关联和样品级质控。')